# 1. Synthetic control: computer occupations

**Question:** after 2022, did computer occupations fall below where they "should" have been?

**The idea:** we can't see tech without AI, so we build a stand-in: a **weighted blend of other
occupations that moved like tech before 2022** (its "synthetic twin"). If tech later falls below
its twin, that gap is the effect.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

panel = pd.read_parquet("../data/processed/panel_occ_age_quarter.parquet")
groups = pd.read_csv("../data/processed/occ_groups.csv").set_index("group")

# Log employment: one column per occupation group, one row per quarter.
emp = np.log(panel.groupby(["t", "group"])["employed"].sum().unstack())
tech = emp["15-12"]                      # computer occupations

## Step 1. Pick the candidates

Only **low-AI-exposure** occupations (below the median on both exposure ratings). If other
AI-exposed jobs were in the blend, an AI effect would hit the twin too and cancel out. Also
only groups big enough to be stable (150,000+ workers) and observed every quarter.

In [2]:
low = (groups.dv_rating_beta < groups.dv_rating_beta.median()) & \
      (groups.human_rating_beta < groups.human_rating_beta.median())
big = groups.employed_2015_19 >= 150_000
candidates = [g for g in groups.index[low & big] if emp[g].notna().all()]
print(len(candidates), "candidate occupations")

36 candidate occupations


## Step 2. Find the blend that most closely follows tech

Choose weights (each ≥ 0, adding up to 1) that make the blend's path as close as possible to
tech's over the fit years. We compare *paths*, not levels (each series minus its own average),
because no mix of smaller jobs can match tech's size, and size isn't the point.

In [3]:
def twin(start, end):
    """Weights fitted on [start, end]; returns the synthetic tech series and the weights."""
    fit = (emp.index >= start) & (emp.index <= end)
    y = tech[fit] - tech[fit].mean()
    X = emp.loc[fit, candidates] - emp.loc[fit, candidates].mean()
    n = len(candidates)
    w = minimize(lambda w: np.sum((y - X @ w) ** 2), np.ones(n) / n, method="SLSQP",
                 bounds=[(0, 1)] * n, constraints={"type": "eq", "fun": lambda w: w.sum() - 1}).x
    synthetic = (emp[candidates] - emp.loc[fit, candidates].mean()) @ w + tech[fit].mean()
    return synthetic, pd.Series(w, candidates)

synthetic, w = twin(2003, 2021.75)
w = w[w > 0.01].sort_values(ascending=False)
pd.DataFrame({"weight": w.round(2), "job (largest in group)": groups.loc[w.index, "title"]})

,weight,job (largest in group)
29-20,0.49,Health Practitioner Support Technologists and ...
39-20,0.33,Nonfarm Animal Caretakers
53-70,0.17,"Laborers and Freight, Stock, and Material Move..."


## Step 3. Compare tech with its twin

In [4]:
ax = pd.DataFrame({"Computer occupations": tech, "Synthetic twin": synthetic}).loc[2003:].plot(figsize=(8, 4))
ax.axvline(2022.75, ls=":", c="gray")
ax.set_ylabel("log employment");

## Step 4. The check that matters: can the twin predict years it wasn't fit on?

Fit only on 2003–2017, then see how far off it is in 2018–2021. If it can't predict years
before AI, its post-2022 gap means nothing.

In [5]:
synthetic_17, _ = twin(2003, 2017.75)
error = lambda a, b: np.sqrt(np.mean((tech.loc[a:b] - synthetic_17.loc[a:b]) ** 2))
print(f"error in the fit years (2003-2017): {error(2003, 2017.75):.1%}")
print(f"error in new years   (2018-2021): {error(2018, 2021.75):.1%}")

error in the fit years (2003-2017): 4.8%
error in new years   (2018-2021): 25.2%


## What it means

**It failed.** The error in new years is about 5× the error in the fit years. The "closest-following"
blend is an odd mix (health technicians, animal caretakers, warehouse laborers) that tracked tech by
coincidence, not because they respond to the same forces. Tech simply grew faster than any mix of
low-exposure jobs.

**What I did next:** moved to *industries* (notebook 2), where payroll data is far less noisy, and
later to designs that pool many units (notebook 3).

The full version (with placebo tests and the dot-com check) is `scripts/synth_computer.py`.
The method: Abadie, Diamond & Hainmueller (2010), "Synthetic Control Methods for Comparative Case Studies."